In [0]:
import pandas as pd
# from loguru import logger
import json

In [0]:
df_bronze = spark.read.table("corp_nonprod.bronze.ops_telcochurn_tab_churn")

## 2. EDA and Cleaning

- Purpose: produce silver table from bronze table with "just enough" cleaning, reformatting and joining so that there are business-friendly tables representing the business operations (e.g. master customers, stores, non-duplicated transactions and cross-reference tables). Reference: https://www.databricks.com/glossary/medallion-architecture
- Common steps:
  - Remove duplicates (demo below)
  - Handle missing values (remove vs filling strategies)
  - Casting types and/or derive more columns from existing ones
  - Standardize column names
  - Enrich the data with other tables

In [0]:
# change all column names to lowercase
df_silver = df_bronze.toDF(*[c.lower() for c in df_bronze.columns])

In [0]:
from pyspark.sql import functions as F

# Find the number of rows with missing values in each column
missing_values_count = df_silver.select(
    [F.count(F.when(F.isnan(c) | F.col(c).isNull(), c)).alias(c) for c in df_silver.columns]
)
display(missing_values_count)

# remove rows with missing values in any column
df_silver = df_silver.dropna(how='any')

In [0]:
# check if there is duplicated customerID, if yes, drop these rows
df_silver = df_silver.dropDuplicates(['customerID'])

# the prints below will be removed in operational pipelines
print("No. rows before dropping duplicates:")
print(df_bronze.count())
print("No. rows after dropping duplicates:")
print(df_silver.count())


In [0]:
# change "seniorcitizen" values: 0 = "No", 1 = "Yes"
df_silver = df_silver.withColumn("seniorcitizen", F.when(df_silver.seniorcitizen == 1, "Yes").otherwise("No"))

In [0]:
# change totalcharges from string to double
df_silver = df_silver.withColumn("totalcharges", F.col("totalcharges").cast("double"))

In [0]:
# change "churn" values: "No" = 0, "Yes" = 1 (numerical labels required for some classifiers)
df_silver = df_silver.withColumn("churn", F.when(df_silver.churn == "Yes", 1).otherwise(0))

In [0]:
display(df_silver, df_silver.limit(5))

## 3. Write to Silver table

In [0]:
# assuming data cleaning was performed, we can write the cleaned table to silver with the following conditions:
# for the first time, the table is not there yet, we can just overwrite:
df_silver.write.mode('overwrite').saveAsTable("corp_nonprod.silver.ops_telcochurn_tab_churn")



In [0]:
# # for subsequent time, assuming incremental data, we update rows with existing customerid, else insert
# from delta.tables import DeltaTable

# # Define the target table
# target_table = "corp_nonprod.silver.ops_telcochurn_tab_churn"

# # Create a DeltaTable object
# delta_table = DeltaTable.forName(spark, target_table)

# # Perform the merge operation
# (delta_table.alias("target")
#  .merge(
#      df_silver.alias("source"),
#      "target.customerid = source.customerid"
#  )
#  .whenMatchedUpdateAll()
#  .whenNotMatchedInsertAll()
#  .execute()
# )